# Model Compilation

### LLM-Based Models

**Goal**: explore frontier and open-source LLMs for Amazon price prediction from product descriptions. 

Data processing and the classical ML benchmarks were developed in notebook `01_eda_and_preprocessing` and `02_classical_models`.

**Methodology**: this notebook evaluates whether LLMs can outperform the best classical approaches, especially those that already benefit from embeddings.

## Index

1. Frontier LLMs
2. Open-source LLMs
3. Fine-tuned Open-source LLMs

In [1]:
# imports
# -------

from pathlib import Path
import pyarrow

import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from huggingface_hub import login
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
import wandb
from datetime import datetime


import sys
sys.path.append(str(Path("../../src").resolve())) # or uv sync - but this will always work instead
from pricing_amazon_products.fine_tuning import QloraTuning
from pricing_amazon_products.inference import (
    LLMPricePredictor,
    evaluate_llm_predictions
)
from pricing_amazon_products.experiments import (
    init_metadata_df,
    update_model_metadata
)
from pricing_amazon_products.config import (
    MODELS_METADATA_PATH, 
    TARGET,
    N_FOLDS,
    RANDOM_STATE,
    SCORING,
    EMBEDDING_MODELS,
    EMBEDDING_SOURCE_COL,
    LLM_PREDICTIONS_DIR
)

load_dotenv(override=True)

c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
embedding_models_df = pd.DataFrame.from_dict(EMBEDDING_MODELS, orient="index")

In [3]:
# load data

split_dir = Path("../../data/splits")

y_train = pd.read_parquet(split_dir  / "y_train.parquet")[["row_id", TARGET]]
y_test = pd.read_parquet(split_dir  / "y_test.parquet")[["row_id", TARGET]]

X_train = pd.read_parquet(split_dir  / "X_train_catalog_content.parquet")
X_test = pd.read_parquet(split_dir  / "X_test_catalog_content.parquet")

# We are going to use the most powerful trained embedding vector
HEAVIER_EMBEDDING_MODEL = embedding_models_df["weight_in_million_params"].idxmax()
emb_name = f'{EMBEDDING_SOURCE_COL}__{HEAVIER_EMBEDDING_MODEL.replace('/', '_')}'
X_train_embeddings = pd.read_parquet(split_dir  / f"X_train_with_embeddings__{emb_name}.parquet").drop(columns='row_id')
X_test_embeddings = pd.read_parquet(split_dir  / f"X_test_with_embeddings__{emb_name}.parquet").drop(columns='row_id')

In [4]:
# Size

print("Train with description shape:", X_train.shape)
print("Train with embeddings shape:", X_train_embeddings.shape)
print("Train target shape:", y_train.shape)
print("Test with description shape:", X_test.shape)
print("Test with embeddings shape:", X_test_embeddings.shape)
print("Test target shape:", y_test.shape)

Train with description shape: (59919, 2)
Train with embeddings shape: (59919, 389)
Train target shape: (59919, 2)
Test with description shape: (14980, 2)
Test with embeddings shape: (14980, 389)
Test target shape: (14980, 2)


In [5]:
X_train.head(2)

,row_id,catalog_content
64402,64402,Item Name: Stonewall Kitchen Sauce Dessert Crm...
22256,22256,Item Name: Food to Live - Organic Medjool Date...


In [6]:
y_test.head(2)

,row_id,log_price
17030,17030,3.688629
69555,69555,1.050822


**LLM model selection for price prediction**

For this project, we are comparing frontier and open-source LLMs to estimate Amazon product prices from `catalog_content`.  

Key trade-offs : predictive quality, API cost, and latency

Interesting model candidates:

| Model | Mode | Context window | Parameters | Cost |
| --- | --- | --- | --- | --- |
| `gemini-3.1-flash-lite` | Cloud/API | 1M tokens | Not disclosed here | $0.25 in / $1.50 out (per 1M) |
| `gemini-3.1-pro` | Cloud/API | 1M tokens | Not disclosed here | $2.00 in / $12.00 out (per 1M) |
| `llama3.2` | Local | 128K tokens | 1B / 3B | Open source |
| `gpt-oss-120b` | Local | 131K tokens | 120B | Open source |

[Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing)  

## Configuration

----

Global settings to keep the experiments reproducible and easy to update:

- `FORCE_OVERWRITE`: if `True`, saved results are always regenerated.
- `MAX_WORDS`: caps the number of words kept from each description before sending it to the LLM.

In [7]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

FORCE_OVERWRITE = False
MAX_WORDS = 125

In [8]:
# TODO: add a contant that overwrites meta if the predicted samples increase

**Model target and evaluator**

- Target: `log_price`
- Metric: RMSE, to be minimized
- Note: Since the target is `log(price)`, RMSE will evaluates error in log space

**Metadata**

This notebook keeps a small experiment log with one row per model run.  
Each row stores the model name, model family, feature label, target, hyperparameters, and evaluation scores so results can be compared later.

The stored metrics will be replaced for a given model if `FORCE_OVERWRITE` is enabled, or if `OVERWRITE_IF_BETTER_CV` is enabled and the new cross-validated score is better than the one already stored.

In [9]:
results_metadata_df = init_metadata_df(overwrite=False)

## 1 | Frontier LLMs

----

**Gemini 3.1 flash lite**

In [ ]:
gemini_model_name = "gemini/gemini-3.1-flash-lite"
# gemini_model_name = "gemini/gemini-2.5-pro"

In [ ]:
predictor = LLMPricePredictor(
    model_name=gemini_model_name,
    max_words=MAX_WORDS,
    checkpoint_every=10,
    sleep_seconds=5,
)

predictions_df = predictor.run(X_test)

In [ ]:
predictions_df = pd.read_parquet(LLM_PREDICTIONS_DIR / "gemini_gemini_3_1_flash_lite.parquet")
predictions_df[predictions_df['status'] == 'ok'].shape

In [ ]:
predictions_df[['row_id', 'catalog_content']].to_csv('claude.csv')

In [ ]:
X_test.to_csv('x_test.csv')

In [ ]:
gemini_results = evaluate_llm_predictions(
    y_test,
    gemini_model_name,
    'Gemini',
    'LLM (catalog_content prompt)'
)
gemini_results

In [ ]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    gemini_results, 
    FORCE_OVERWRITE
)

## 2 | Open-source LLMs

----

In [ ]:
llama_model_name = "ollama_chat/llama3.2"

In [ ]:
predictor = LLMPricePredictor(
    model_name=llama_model_name,
    max_words=MAX_WORDS,
    checkpoint_every=10,
)

predictions_df = predictor.run(X_test)

In [ ]:
llama_results = evaluate_llm_predictions(
    y_test,
    llama_model_name,
    'Llama',
    'LLM (catalog_content prompt)'
)
llama_results

In [ ]:
predictions_df = pd.read_parquet(LLM_PREDICTIONS_DIR / "ollama_chat_llama3_2.parquet")
predictions_df[predictions_df['status'] == 'ok'].shape

In [ ]:
predictions_df = pd.read_parquet(LLM_PREDICTIONS_DIR / "ollama_chat_llama3_2.parquet")
predictions_df[predictions_df['status'] == 'ok'].head()

In [ ]:
y_test.head(5)

In [ ]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    llama_results, 
    FORCE_OVERWRITE
)

## 3. Fine-tuned Open-source LLMs

----

In [10]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

PILOT_MODE = True
BASE_MODEL = "meta-llama/Llama-3.2-1B"
PROJECT_NAME = "amazon_price_prediction"
HF_USER = "GONVF"
LOG_TO_WANDB = True

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

In [11]:
# HF login
hf_token = os.getenv("HF_TOKEN")
if hf_token is None:
    raise ValueError("HF_TOKEN not found in environment variables")

login(token=hf_token, add_to_git_credential=True)

# Log in to Weights & Biases
wandb_api_key = os.getenv("WANDB_API_KEY")
if wandb_api_key is None:
    raise ValueError("WANDB_API_KEY not found in environment variables")

os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: gonvillalon (gonvillalon-universidad-de-navarra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
qt = QloraTuning(
    epochs=2,
    batch_size=32,
    max_sequence_length=128,
    gradient_accumulation_steps=1,
    quant_4_bit=True,
    lora_r=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    learning_rate=1e-3,
    warmup_ratio=0.01,
    lr_scheduler_type="cosine",
    weight_decay=0.001,
    optimizer="paged_adamw_32bit",
    val_size=500,
    log_steps=5,
    save_steps=100,
    log_to_wandb=True,
    project_name=PROJECT_NAME,
    project_run_name=PROJECT_RUN_NAME,
    run_name=RUN_NAME,
    hub_model_name=HUB_MODEL_NAME,
    base_model_name=BASE_MODEL
)
# qt.run_model(X_train, y_train, PILOT_MODE)

In [ ]:
hub_model_name = "GONVF/amazon_price_prediction-2026-06-18_22.17.55-lite"

predictor = LLMPricePredictor(
    model_name=BASE_MODEL,
    max_words=MAX_WORDS,
    checkpoint_every=10,
    hub_model_name=hub_model_name,
    backend='hf',
    quant_4_bit=True
)

predictions_df = predictor.run(X_test)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 146/146 [03:28<00:00,  1.43s/it]
c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gonza\.cache\huggingface\hub\models--

Memory footprint: 1039.3 MB


c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Checkpoint saved at 10 predictions


c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\gonza\Documents\pricing-amazon-products\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Checkpoint saved at 20 predictions
Checkpoint saved at 30 predictions
Checkpoint saved at 40 predictions
Checkpoint saved at 50 predictions
Checkpoint saved at 60 predictions


In [10]:

predictions_df = pd.read_parquet(LLM_PREDICTIONS_DIR / "meta_llama_Llama_3_2_1B__GONVF_amazon_price_prediction_2026_06_18_22_17_55_lite.parquet")
predictions_df[predictions_df['status'] == 'ok'].shape

(15, 7)

In [13]:
predictions_df.head()

,row_id,catalog_content,model_name,raw_response,pred_price,pred_log_price,status
0,17030,Item Name: De La Rosa Organic Red Wine Vinegar...,meta-llama/Llama-3.2-1B,��𝐨𝐧,NaN,NaN,bad_parse
1,69555,Item Name: Tru Fru Dark Chocolate Covered Hype...,meta-llama/Llama-3.2-1B,1-pound package\nBullet Point,1.0,0.0,ok
2,52078,Item Name: Pasta Roni Fettuccini Alfredo 4.7 o...,meta-llama/Llama-3.2-1B,Item Name: Pasta Roni Fett,NaN,NaN,bad_parse
3,45135,Item Name: Morton & Bassett Cinnamon Stix Org\...,meta-llama/Llama-3.2-1B,Item Name: Morton & Bassett C,NaN,NaN,bad_parse
4,32638,"Item Name: Jalapeno Seasoning Powder, Jalapeno...",meta-llama/Llama-3.2-1B,favorite recipes.,NaN,NaN,bad_parse


In [ ]:
qlora_results = evaluate_llm_predictions(
    y_test,
    llama_model_name,
    'QLoRA',
    'LLM (catalog_content prompt)'
)
llama_results

In [ ]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    qlora_results, 
    FORCE_OVERWRITE
)

## Next steps

----

To improve this module, especially the LLM-based workflows:

1. Improve text compression before inference:
- Replace the current naive truncation strategy with a more informed compression step. Right now, `LLMPricePredictor` keeps only the first 80% of the words and the last 20%, which can discard informative parts of the product description and reduce model performance.
- A better alternative would be to use an intermediate model or summarization step that condenses the text while preserving the most relevant attributes for price prediction.
- If using an extra model for summarization is out of the budget, we could compose a text file by uniting `item_name`, `value`, and `unit` columns, which were already processed